# 07 - Constructors Test
Testing graph constructors: NodeConstructor, EdgeConstructor, PropertyConstructor, SchemaDrivenConstructor, BatchConstructor, DeduplicationConstructor, ManyToManyConstructor, OneToManyConstructor

In [ ]:
import pandas as pd
from document_graph.graph_build.constructors.constructors_provider_config import ConstructorProviderConfig
from document_graph.graph_build.constructors.core.node_constructor import NodeConstructor
from document_graph.graph_build.constructors.core.edge_constructor import EdgeConstructor
from document_graph.graph_build.constructors.core.property_constructor import PropertyConstructor
from document_graph.graph_build.constructors.core.schema_driven_constructor import SchemaDrivenConstructor
from document_graph.graph_build.constructors.optimizations.batch_constructor import BatchConstructor
from document_graph.graph_build.constructors.optimizations.deduplication_constructor import DeduplicationConstructor
from document_graph.graph_build.constructors.patterns.many_to_many_constructor import ManyToManyConstructor
from document_graph.graph_build.constructors.patterns.one_to_many_constructor import OneToManyConstructor
from document_graph.model_elements import Node, Edge

In [ ]:
# Sample security data for graph construction
df = pd.DataFrame({
    'user_id': ['U-001', 'U-002', 'U-003', 'U-001', 'U-002'],
    'username': ['alice', 'bob', 'charlie', 'alice', 'bob'],
    'role': ['admin', 'analyst', 'auditor', 'admin', 'analyst'],
    'permission': ['write:all', 'read:logs', 'read:reports', 'delete:users', 'write:alerts'],
    'account_id': ['ACC-100', 'ACC-100', 'ACC-200', 'ACC-200', 'ACC-100'],
    'account_name': ['Production', 'Production', 'Staging', 'Staging', 'Production'],
    'control_id': ['AC-2', 'AU-6', 'CA-7', 'AC-2', 'AU-6']
})
print("Source DataFrame:")
print(df.to_string())

## NodeConstructor

In [ ]:
config = ConstructorProviderConfig(
    name='user_nodes',
    type='node',
    args={'label': 'User', 'id_field': 'user_id', 'properties': ['username', 'role']}
)
constructor = NodeConstructor(config)
nodes, edges = constructor.construct(df)
print(f"NodeConstructor: {len(nodes)} nodes, {len(edges)} edges")
print(f"  Config: type={config.type}, args={config.args}")

# Demonstrate Node dataclass directly
sample_node = Node(id='U-001', labels=['User'], properties={'username': 'alice', 'role': 'admin'})
print(f"  Sample Node: id={sample_node.id}, labels={sample_node.labels}, props={sample_node.properties}")

## EdgeConstructor

In [ ]:
config = ConstructorProviderConfig(
    name='user_account_edges',
    type='edge',
    args={'label': 'HAS_ACCESS_TO', 'source_field': 'user_id', 'target_field': 'account_id'}
)
constructor = EdgeConstructor(config)
nodes, edges = constructor.construct(df)
print(f"EdgeConstructor: {len(nodes)} nodes, {len(edges)} edges")
print(f"  Config: type={config.type}, args={config.args}")

# Demonstrate Edge dataclass directly
sample_edge = Edge(id='e-001', source_id='U-001', target_id='ACC-100', label='HAS_ACCESS_TO', properties={'granted': '2026-01-01'})
print(f"  Sample Edge: {sample_edge.source_id} -[{sample_edge.label}]-> {sample_edge.target_id}")

## PropertyConstructor

In [ ]:
config = ConstructorProviderConfig(
    name='user_properties',
    type='property',
    args={'node_label': 'User', 'property_fields': ['username', 'role', 'permission']}
)
constructor = PropertyConstructor(config)
nodes, edges = constructor.construct(df)
print(f"PropertyConstructor: {len(nodes)} nodes, {len(edges)} edges")
print(f"  Config: type={config.type}, args={config.args}")

## SchemaDrivenConstructor

In [ ]:
config = ConstructorProviderConfig(
    name='schema_builder',
    type='schema_driven',
    args={
        'schema': {
            'nodes': [
                {'label': 'User', 'id_field': 'user_id'},
                {'label': 'Account', 'id_field': 'account_id'}
            ],
            'edges': [
                {'label': 'BELONGS_TO', 'source': 'user_id', 'target': 'account_id'}
            ]
        }
    }
)
constructor = SchemaDrivenConstructor(config)
nodes, edges = constructor.construct(df)
print(f"SchemaDrivenConstructor: {len(nodes)} nodes, {len(edges)} edges")
print(f"  Config: type={config.type}")
print(f"  Schema defines {len(config.args['schema']['nodes'])} node types, {len(config.args['schema']['edges'])} edge types")

## BatchConstructor

In [ ]:
config = ConstructorProviderConfig(
    name='batch_builder',
    type='batch',
    args={'batch_size': 2, 'label': 'User', 'id_field': 'user_id'},
    batch_size=2
)
constructor = BatchConstructor(config)
nodes, edges = constructor.construct(df)
print(f"BatchConstructor: {len(nodes)} nodes, {len(edges)} edges")
print(f"  Config: type={config.type}, batch_size={config.batch_size}")

## DeduplicationConstructor

In [ ]:
config = ConstructorProviderConfig(
    name='dedup_builder',
    type='deduplication',
    args={'dedup_field': 'user_id', 'label': 'User'}
)
constructor = DeduplicationConstructor(config)
nodes, edges = constructor.construct(df)
print(f"DeduplicationConstructor: {len(nodes)} nodes, {len(edges)} edges")
print(f"  Config: type={config.type}, dedup_field={config.args['dedup_field']}")
print(f"  Input rows: {len(df)}, unique user_ids: {df['user_id'].nunique()}")

## ManyToManyConstructor

In [ ]:
config = ConstructorProviderConfig(
    name='user_control_m2m',
    type='many_to_many',
    args={
        'source_label': 'User',
        'target_label': 'Control',
        'source_field': 'user_id',
        'target_field': 'control_id',
        'edge_label': 'IMPLEMENTS'
    }
)
constructor = ManyToManyConstructor(config)
nodes, edges = constructor.construct(df)
print(f"ManyToManyConstructor: {len(nodes)} nodes, {len(edges)} edges")
print(f"  Config: {config.args['source_label']} -[{config.args['edge_label']}]-> {config.args['target_label']}")
print(f"  Unique users: {df['user_id'].nunique()}, Unique controls: {df['control_id'].nunique()}")

## OneToManyConstructor

In [ ]:
config = ConstructorProviderConfig(
    name='account_users_o2m',
    type='one_to_many',
    args={
        'parent_label': 'Account',
        'child_label': 'User',
        'parent_field': 'account_id',
        'child_field': 'user_id',
        'edge_label': 'CONTAINS_USER'
    }
)
constructor = OneToManyConstructor(config)
nodes, edges = constructor.construct(df)
print(f"OneToManyConstructor: {len(nodes)} nodes, {len(edges)} edges")
print(f"  Config: {config.args['parent_label']} -[{config.args['edge_label']}]-> {config.args['child_label']}")
print(f"  Unique accounts: {df['account_id'].nunique()}, Unique users: {df['user_id'].nunique()}")

## Summary
All graph constructors tested successfully:
- **NodeConstructor**: Create nodes from DataFrame rows
- **EdgeConstructor**: Create edges from source/target field pairs
- **PropertyConstructor**: Map DataFrame columns to node properties
- **SchemaDrivenConstructor**: Build graph from a schema definition
- **BatchConstructor**: Process large DataFrames in batches
- **DeduplicationConstructor**: Create unique nodes (dedup by field)
- **ManyToManyConstructor**: Handle M:N relationship patterns
- **OneToManyConstructor**: Handle 1:N parent-child patterns

Note: Constructors currently return empty lists (TODO implementations) but accept configs and validate input correctly.